# Banana Lifeline — Training on a Colab T4

This notebook is a **driver, not a codebase**. Every cell calls into the installed
`banana_lifeline` package, which is version-controlled, type-checked, and tested. Training
logic deliberately does not live in notebook cells, where it cannot be reviewed or tested
and quietly forks from the repo.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

### What this run produces

Two models trained with an identical recipe, differing only in how images were split:

| Run | Split | What it tells you |
|---|---|---|
| `clean` | Group-aware, leakage-free | **The real performance.** |
| `original` | The shipped split | What the contaminated split claims. |

The shipped split has **622 of 1,033 held-out images (60%) with a near-identical twin in
training**, so its accuracy is largely memorisation. Training both is what turns that from
an assertion into a measured result.

## 1. Confirm the GPU

T4 is Turing generation: it has fp16 tensor cores but **no bf16**. The training config sets
`amp_dtype: float16` for that reason — it is a hardware fact, not a preference. If you are
given an A100 or L4 instead, bf16 becomes available and is more numerically forgiving.

In [1]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

name, memory.total [MiB], compute_cap
Tesla T4, 15360 MiB, 7.5


## 2. Get the code and the data

Set `REPO_URL` to your remote. The **cache**, not the raw corpus, is what gets uploaded:
3.5 GB of phone photos becomes 136 MB at 512px, which is the difference between a
multi-hour upload and a one-minute one.

Two options for the data — pick one:

- **Google Drive** (recommended): upload `ai/data/cache/` and `ai/data/manifests/` to Drive
  once, then mount. Survives runtime restarts.
- **Direct upload**: zip `ai/data/` locally and upload it each session.

In [8]:
REPO_URL = "https://github.com/Muhammad-Saaaad/Banana-Lifeline.git"  # <-- set this
USE_DRIVE = True

import os, pathlib, subprocess

if not pathlib.Path("Banana-Lifeline").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)

AI_ROOT = pathlib.Path("Banana-Lifeline/ai").resolve()

# Every module resolves paths through this, so nothing else needs configuring.
os.environ["BANANA_AI_ROOT"] = str(AI_ROOT)
print("AI root:", AI_ROOT)

AI root: /content/Banana-Lifeline/ai


In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    # Expects banana_data/cache/ and banana_data/manifests/ in your Drive root.
    SOURCE = pathlib.Path("/content/drive/MyDrive/banana_data")
    !mkdir -p "$AI_ROOT/data"
    !cp -r "$SOURCE/cache" "$AI_ROOT/data/"
    !cp -r "$SOURCE/manifests" "$AI_ROOT/data/"
else:
    from google.colab import files
    print("Upload a zip of ai/data/ (cache + manifests)")
    uploaded = files.upload()
    for name in uploaded:
        !unzip -q "$name" -d "$AI_ROOT/"

!ls "$AI_ROOT/data/manifests"
!echo "cached images: $(ls $AI_ROOT/data/cache | wc -l)"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Install

Colab ships torch already; only the extras are needed. `--no-deps` on the package itself
avoids pip resolving a different torch build over Colab's CUDA-matched one.

In [ ]:
%pip install -q timm albumentations grad-cam
%pip install -q -e "$AI_ROOT" --no-deps

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f"{torch.cuda.get_device_name(0)}  sm_{major}{minor}")
    print("bf16 supported:", torch.cuda.is_bf16_supported(), "(False on T4, as expected)")

## 4. Sanity check the split before spending GPU time

This asserts no scene spans two splits. If it fails, **stop** — every metric downstream
would be meaningless, and that is exactly the bug this project exists to have fixed.

In [ ]:
import pandas as pd
from banana_lifeline.config import get_paths, load_data_config, load_train_config
from banana_lifeline.data.dataset import load_manifest, manifest_path
from banana_lifeline.data.split import verify_no_leakage

paths = get_paths()
clean = load_manifest(manifest_path(paths, "clean"))
verify_no_leakage(clean)

print("Leakage check passed.\n")
print(pd.crosstab(clean["label"], clean["split"]))
print("\nscenes per split:")
print(clean.groupby("split")["cluster_id"].nunique())

## 5. Train on the clean split

Roughly 10–20 minutes on a T4 at 384px. Progress prints macro-F1 and **panama recall** per
epoch — watch panama recall, not accuracy: sigatoka outnumbers panama ~4:1, so accuracy can
look healthy while the model quietly ignores the disease that kills the plant.

**If you hit CUDA OOM:** drop `batch_size` to 16 and set `grad_accum_steps=2`, which keeps
the effective batch size unchanged.

A checkpoint is written every epoch, so a Colab disconnect costs one epoch. Resume with
`resume=run_dir / "last.pt"`.

In [ ]:
from banana_lifeline.vision.train import train

data_cfg = load_data_config()
train_cfg = load_train_config()

print(f"{train_cfg.model.architecture} @ {train_cfg.image_size}px, "
      f"batch {train_cfg.batch_size}x{train_cfg.grad_accum_steps}, "
      f"amp={train_cfg.amp_dtype}\n")

clean_run = train(clean, train_cfg, data_cfg, paths, run_name="clean")

## 6. Train on the contaminated split

Same recipe, same hyperparameters, same quarantined scenes. **The only variable is the
split**, which is what makes the comparison mean something: if the two runs differed in more
than one way, a gap in accuracy would have several possible causes and would prove nothing.

Expect this run to look excellent. That is the point — it is the number the broken split
would have let you report.

In [ ]:
original = load_manifest(manifest_path(paths, "original"))
original_run = train(original, train_cfg, data_cfg, paths, run_name="original")

## 7. Evaluate, calibrate, and compare

Calibration is fitted on **validation**, never on test — fitting a threshold on test data
leaks it and inflates the result.

In [ ]:
from banana_lifeline.vision.calibrate import calibrate
from banana_lifeline.vision.evaluate import (
    background_ablation, comparison_report, predict, write_evaluation,
)
from banana_lifeline.vision.model import load_checkpoint

loaded = load_checkpoint(clean_run / "best.pt")
test_rows = clean[clean["split"] == "test"]
val_rows = clean[clean["split"] == "val"]

clean_out = predict(loaded.model, test_rows, loaded.train_config, data_cfg, paths)
print("clean  :", clean_out.metrics.headline())

original_loaded = load_checkpoint(original_run / "best.pt")
original_out = predict(
    original_loaded.model,
    original[original["split"] == "test"],
    original_loaded.train_config, data_cfg, paths,
)
print("claimed:", original_out.metrics.headline())

val_out = predict(loaded.model, val_rows, loaded.train_config, data_cfg, paths)
calibration = calibrate(val_out.logits, val_out.labels)
print("\n", calibration.summary())

### Background shortcut check

Every photograph in this corpus comes from one site on one day, so sky, soil, and lighting
barely vary between classes and a model can score well by reading context rather than
pathology. Re-running on a tight centre crop removes most of that context.

A **small drop** means the model reads the leaf. A **large drop** means much of the score
will not survive a different farm or season.

In [ ]:
ablation = background_ablation(loaded.model, test_rows, loaded.train_config, data_cfg, paths)
drop = clean_out.metrics.macro_f1 - ablation.macro_f1
print(f"full frame {clean_out.metrics.macro_f1:.3f} -> centre crop {ablation.macro_f1:.3f}")
print(f"drop {drop:.3f} —", "reading the leaf" if drop < 0.10 else "LIKELY SHORTCUT LEARNING")

In [ ]:
report = comparison_report(
    clean_out.metrics, original_out.metrics,
    ablation=ablation, calibration=calibration.to_dict(),
)
write_evaluation(report, {
    "clean": clean_out.metrics.to_dict(),
    "original": original_out.metrics.to_dict(),
    "ablation": ablation.to_dict(),
    "calibration": calibration.to_dict(),
}, paths)

from IPython.display import Markdown, display
display(Markdown(report))

## 8. Grad-CAM — see what the model looks at

Heat on lesions and leaf tissue is the model working. Heat on sky, soil, or a hand means the
score is borrowed from context. Files prefixed `WRONG_` are the misclassifications, which is
where shortcut learning shows up most clearly.

In [ ]:
from banana_lifeline.vision.explain import generate_overlays

overlay_dir = generate_overlays(loaded.model, test_rows, loaded.train_config, data_cfg, paths)

import matplotlib.pyplot as plt
from PIL import Image

files_ = sorted(overlay_dir.glob("*.jpg"))[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, path in zip(axes.flatten(), files_):
    ax.imshow(Image.open(path))
    ax.set_title(path.stem.replace("_", " ")[:44], fontsize=8)
    ax.axis("off")
for ax in axes.flatten()[len(files_):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 9. Save results back to Drive

Colab wipes local storage when the runtime ends. Weights, metrics, and the report need to
leave the machine.

In [ ]:
if USE_DRIVE:
    DEST = pathlib.Path("/content/drive/MyDrive/banana_runs")
    DEST.mkdir(parents=True, exist_ok=True)
    !cp -r "$AI_ROOT/runs" "$DEST/"
    !cp -r "$AI_ROOT/reports" "$DEST/"
    print("saved to", DEST)
else:
    !cd "$AI_ROOT" && zip -qr /content/banana_results.zip runs reports
    from google.colab import files
    files.download("/content/banana_results.zip")

---

## What to do with these numbers

**Report the clean figures.** The contaminated ones exist to show what the broken split
would have let you claim, and the gap between them is the finding — not an embarrassment.
"We found a 60% leakage bug, fixed it, and measured the real cost" is a stronger result for
a report or viva than a high accuracy number that would collapse on a farmer's photo.

Things worth checking before calling this done:

- **Panama recall.** The smallest class (329 training scenes) and the most costly to miss.
  If it is weak, that is the honest headline, and it is an argument for collecting more
  panama data rather than for changing the model.
- **The confusion matrix.** Panama/sigatoka confusion is expected — 16 of the 18 contested
  scenes were ruled undecidable from a leaf photo by review, which is itself evidence that
  the distinction is not always visible in this kind of image.
- **The ablation drop.** Large means the model leans on background, and the number will not
  transfer to another farm.

Next: Phase 3 adds out-of-distribution rejection, so a photo of a goat or a blurry thumb
gets an honest "I can't read this" instead of a confident diagnosis.